<a href="https://colab.research.google.com/github/WiseSeaSnake/CZN_AD/blob/main/Itog_proba.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import pandas as pd


import pandas as pd

file_path = "кейс 40.xlsx"
sheets = pd.read_excel(file_path, sheet_name=None, engine="openpyxl")

for sheet_name, df in sheets.items():
    # Делаем безопасное имя переменной
    var_name = "df_" + sheet_name.lower().replace(" ", "_").replace("-", "_")
    globals()[var_name] = df

print('\nProduct \n')
print(df_product.head())
print('\nRegion \n')
print(df_region.head())
print('\nSales \n')
print(df_sales.head())
print('\nState \n')
print(df_state.head())

print('\nФильтруем\n')
df_filtered_product_category = df_product[df_product['ProductCategory'] == 'Glider']
print(df_filtered_product_category)

df_filtered_product_retail_price = df_product[df_product['RetailPrice'] > 100]
print(df_filtered_product_retail_price)






Product 

   ProductID     ProductSKU                ProductName ProductCategory  \
0          1  1010-GL120-3C  Trainer - Tailspin GL-120          Glider   
1          2  1010-GL155-4C  Trainer - Tailspin GL-155          Glider   
2          3   2030-PCUB-3C        Piper Cub 3 Channel         Trainer   
3          4   2030-PCUB-4C        Piper Cub 4 Channel         Trainer   
4          5    2050-P47-4C              P47 4 Channel         Warbird   

  ItemGroup KitType  Channels   Demographic  RetailPrice  
0  Airplane     RTF         3        Novice        79.15  
1  Airplane     RTF         4  Intermediate       216.65  
2  Airplane     RTF         3      Beginner        84.65  
3  Airplane     RTF         4  Intermediate       193.55  
4  Airplane     RTF         4  Intermediate        98.95  

Region 

   RegionID         RegionName
0         1            Midwest
1         2        New England
2         3          Northeast
3         4  Pacific Northwest
4         5           Sou

In [12]:
# ---------- 2. Объединяем всё в один плоский датафрейм ----------
# Sales -> Product
df = df_sales.merge(df_product, on="ProductID", how="left")

# Sales.CustomerStateID -> State -> Region
df = df.merge(
    df_state.rename(columns={"StateID": "CustomerStateID"}),
    on="CustomerStateID", how="left"
)
df = df.merge(df_region, on="RegionID", how="left")

print("Итоговый датафрейм:")
print(df.head(), "\n")
print("Колонки:", list(df.columns), "\n")

Итоговый датафрейм:
  OrderNumber  OrderDate   ShipDate  CustomerStateID  ProductID  Quantity  \
0  TT00017434 2017-01-01 2017-01-09               14          3         1   
1  TT00017445 2017-01-02 2017-01-04               13          3         1   
2  TT00017456 2017-01-06 2017-01-09               35          3         1   
3  TT00017467 2017-01-08 2017-01-11                3          3         1   
4  TT00017478 2017-01-08 2017-01-10                5          3         1   

   UnitPrice  DiscountAmount PromotionCode    ProductSKU  ... ProductCategory  \
0      69.95             0.0           NaN  2030-PCUB-3C  ...         Trainer   
1      69.95             0.0           NaN  2030-PCUB-3C  ...         Trainer   
2      69.95             0.0           NaN  2030-PCUB-3C  ...         Trainer   
3      69.95             0.0           NaN  2030-PCUB-3C  ...         Trainer   
4      69.95             0.0           NaN  2030-PCUB-3C  ...         Trainer   

  ItemGroup KitType Channels  

In [14]:
import numpy as np
# ---------- 3. ПЕРЕКРЁСТНЫЕ ПРИЗНАКИ ----------

# --- 3.1 Категориальные кроссы (cat × cat) ---
cat_pairs = [
    ("ProductCategory", "KitType"),
    ("ProductCategory", "Demographic"),
    ("ItemGroup",       "Demographic"),
    ("KitType",         "Demographic"),
    ("ItemGroup",       "ProductCategory"),
    ("RegionName",      "ProductCategory"),
    ("RegionName",      "Demographic"),
    ("StateName",       "ProductCategory"),
]

for a, b in cat_pairs:
    col = f"{a}_x_{b}"
    df[col] = df[a].astype(str) + "_" + df[b].astype(str)

# --- 3.2 Категориальный кросс по трём полям ---
df["ItemGroup_x_KitType_x_Demographic"] = (
    df["ItemGroup"].astype(str) + "_" +
    df["KitType"].astype(str)   + "_" +
    df["Demographic"].astype(str)
)

# --- 3.3 Числовые взаимодействия (num × num) ---
df["RetailPrice_x_Quantity"]  = df["RetailPrice"] * df["Quantity"]
df["UnitPrice_x_Quantity"]    = df["UnitPrice"]   * df["Quantity"]
df["Channels_x_RetailPrice"]  = df["Channels"]    * df["RetailPrice"]
df["Channels_x_UnitPrice"]    = df["Channels"]    * df["UnitPrice"]

df["RetailPrice_div_Channels"] = df["RetailPrice"] / df["Channels"].replace(0, np.nan)
df["UnitPrice_div_Channels"]   = df["UnitPrice"]   / df["Channels"].replace(0, np.nan)
df["RetailPrice_minus_UnitPrice"] = df["RetailPrice"] - df["UnitPrice"]
df["RetailPrice_div_UnitPrice"]   = df["RetailPrice"] / df["UnitPrice"].replace(0, np.nan)

# --- 3.4 Число × категория (target-подобные признаки) ---
# Средняя RetailPrice по кроссу категорий
for a, b in cat_pairs:
    col = f"{a}_x_{b}"
    df[f"avg_RetailPrice_by_{col}"] = df.groupby(col)["RetailPrice"].transform("mean")
    df[f"avg_Channels_by_{col}"]    = df.groupby(col)["Channels"].transform("mean")

# --- 3.5 Признаки по времени ---
df["OrderYear"]     = df["OrderDate"].dt.year
df["OrderMonth"]    = df["OrderDate"].dt.month
df["OrderWeekday"]  = df["OrderDate"].dt.dayofweek
df["ShipDays"]      = (df["ShipDate"] - df["OrderDate"]).dt.days
df["OrderMonth_x_ProductCategory"] = (
    df["OrderMonth"].astype(str) + "_" + df["ProductCategory"].astype(str)
)

# --- 3.6 Бинарные (one-hot) кроссы: топ-N категорий ---
# Пример: 3 самых частых значения Demographic × KitType
for col in ["ProductCategory", "KitType", "Demographic", "ItemGroup"]:
    top = df[col].value_counts().head(3).index
    for v in top:
        safe = str(v).replace(" ", "_").replace("-", "_")
        df[f"is_{col}_{safe}"] = (df[col] == v).astype(int)

print("Датафрейм с перекрёстными признаками:")
print(df.head().T, "\n")
print("Всего колонок:", df.shape[1])
print([c for c in df.columns if "_x_" in c or "_div_" in c or "_minus_" in c])

Датафрейм с перекрёстными признаками:
                                               0                    1  \
OrderNumber                           TT00017434           TT00017445   
OrderDate                    2017-01-01 00:00:00  2017-01-02 00:00:00   
ShipDate                     2017-01-09 00:00:00  2017-01-04 00:00:00   
CustomerStateID                               14                   13   
ProductID                                      3                    3   
...                                          ...                  ...   
is_Demographic_Professional                    0                    0   
is_Demographic_Intermediate                    0                    0   
is_Demographic_Advanced                        0                    0   
is_ItemGroup_Helicopter                        0                    0   
is_ItemGroup_Airplane                          1                    1   

                                               2                    3  \
OrderNumber 